# Brand Verification — OCR primary, NaFlex fallback

Pipeline step: each OWLv2 text-guided detection is scored for brand presence.

1. **EasyOCR** reads text on the crop and fuzzy-matches against brand keywords from `zbiotics.json`.
2. If OCR returned **no text at all**, fall back to **SigLIP 2 NaFlex** text-image similarity to brand prompts.
3. The combined score is reported per detection.

Eval at the end compares the combined brand score against OWLv2 text's own detection confidence on the same boxes, using the same metrics as `SSL_EVAL.ipynb`: per-category means, separation, AUC, top-1.

In [1]:
%pip install --quiet easyocr rapidfuzz "numpy<2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from rapidfuzz import fuzz
from transformers import AutoModel, AutoProcessor

import easyocr


REPO = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
OUTDIR = REPO / "experiments" / "stage1"
OUTDIR.mkdir(parents=True, exist_ok=True)

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float32

print("device:", DEVICE)


def free_memory():
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()


frame_cache = {}


def get_frame(path):
    if path not in frame_cache:
        frame_cache[path] = Image.open(path).convert("RGB")
    return frame_cache[path]


def crop_box(img, box):
    x0, y0, x1, y1 = (int(v) for v in box)
    x0 = max(0, x0)
    y0 = max(0, y0)
    x1 = min(img.width, x1)
    y1 = min(img.height, y1)
    if x1 <= x0 or y1 <= y0:
        return None
    return img.crop((x0, y0, x1, y1))


# Load OVOD detections — fail fast if missing
DETECTIONS_PATH = OUTDIR / "ovod_detections.jsonl"
if not DETECTIONS_PATH.exists():
    raise FileNotFoundError(
        f"{DETECTIONS_PATH} missing — run OVOD_eval.ipynb first"
    )

OVOD_MODEL = "owlv2_text_guided"
all_detections = [json.loads(line) for line in open(DETECTIONS_PATH)]
detections = [d for d in all_detections if d["model"] == OVOD_MODEL]
print(f"loaded {len(detections)} {OVOD_MODEL} detections (of {len(all_detections)} total)")

# Brand definitions
BRAND_QUERIES = [
    "the ZBiotics brand logo, a large angular black Z mark",
    "ZBIOTICS text on a small frosted glass bottle",
    "pre-alcohol probiotic drink",
    "Z logo",
    "ZBIOTICS",
    "small bottle with black Z logo and orange text",
]

with open(REPO / "data" / "references" / "zbiotics.json") as f:
    meta = json.load(f)

BRAND_KEYWORDS = meta.get("brand_text_keywords", [
    "zbiotics", "pre-alcohol", "probiotic",
])

print("brand queries:", len(BRAND_QUERIES), "| brand keywords:", BRAND_KEYWORDS)

# Per-detection records, populated by NaFlex then OCR cells
results = []

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps
loaded 46 owlv2_text_guided detections (of 1775 total)
brand queries: 6 | brand keywords: ['zbiotics', 'z biotics', 'pre-alcohol', 'pre alcohol', 'probiotic', 'probiotic drink']


In [3]:
# NaFlex scoring — text-image similarity, max over brand queries
naflex_id = "google/siglip2-base-patch16-naflex"
naflex_proc = AutoProcessor.from_pretrained(naflex_id)
naflex_model = AutoModel.from_pretrained(naflex_id, torch_dtype=DTYPE).to(DEVICE).eval()


@torch.no_grad()
def naflex_text_embed(texts):
    x = naflex_proc(text=texts, return_tensors="pt", padding="max_length", truncation=True).to(DEVICE)
    out = naflex_model.get_text_features(**x)
    if not isinstance(out, torch.Tensor):
        out = out.pooler_output if hasattr(out, "pooler_output") else out
    return F.normalize(out, dim=-1)


@torch.no_grad()
def naflex_image_embed(img):
    x = naflex_proc(images=img, return_tensors="pt").to(DEVICE)
    out = naflex_model.get_image_features(**x)
    if not isinstance(out, torch.Tensor):
        out = out.pooler_output if hasattr(out, "pooler_output") else out
    return F.normalize(out, dim=-1)


query_embs = naflex_text_embed(BRAND_QUERIES)

for d in detections:
    img = get_frame(d["frame_path"])
    crop = crop_box(img, d["box"])
    if crop is None:
        continue
    crop_emb = naflex_image_embed(crop)
    sims = (crop_emb @ query_embs.T).squeeze(0).cpu().tolist()
    results.append({
        "label": d["label"],
        "category": d["category"],
        "frame": d["frame"],
        "frame_path": d["frame_path"],
        "box": d["box"],
        "ovod_score": d["score"],
        "naflex_score": float(max(sims)),
    })

print("scored", len(results), "detections with NaFlex")

del naflex_model, naflex_proc, query_embs
free_memory()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 408/408 [00:00<00:00, 11867.05it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/Users/alexwang/Documents/GitHub/storymode/data/frames/positive_easy/zbiotics1.png'

In [ ]:
# OCR scoring — fuzzy-match detected text against brand keywords
reader = easyocr.Reader(["en"], gpu=False)


def ocr_score_for_crop(crop):
    """Return (max_fuzzy_match in [0, 1], list of detected text strings)."""
    arr = np.array(crop)
    detected = reader.readtext(arr, detail=0)
    if not detected:
        return 0.0, []
    best = 0.0
    for text in detected:
        for kw in BRAND_KEYWORDS:
            ratio = fuzz.partial_ratio(text.lower(), kw.lower()) / 100.0
            if ratio > best:
                best = ratio
    return best, detected


for r in results:
    img = get_frame(r["frame_path"])
    crop = crop_box(img, r["box"])
    if crop is None:
        r["ocr_score"] = 0.0
        r["ocr_texts"] = []
        continue
    score, texts = ocr_score_for_crop(crop)
    r["ocr_score"] = score
    r["ocr_texts"] = texts

n_with_text = sum(1 for r in results if r["ocr_texts"])
print(f"detections with any OCR text: {n_with_text}/{len(results)} (rest will use NaFlex fallback)")

In [ ]:
# Combine scores: OCR primary, NaFlex fallback if OCR returned no text.
# Then evaluate combined score head-to-head against OWLv2 text's own confidence
# using the same metrics as SSL_EVAL (separation, AUC, top-1, category means).
for r in results:
    r["combined_score"] = r["ocr_score"] if len(r["ocr_texts"]) > 0 else r["naflex_score"]


def auc(pos_scores, neg_scores):
    if not pos_scores or not neg_scores:
        return float("nan")
    pos = np.array(pos_scores)
    neg = np.array(neg_scores)
    diffs = pos[:, None] - neg[None, :]
    return float((np.sum(diffs > 0) + 0.5 * np.sum(diffs == 0)) / (len(pos) * len(neg)))


def top1_correctness(rows, score_key):
    """Per-frame top-1 score for positives must exceed all negative-frame top-1 scores."""
    by_frame = {}
    for r in rows:
        key = (r["label"], r["category"], r["frame"])
        by_frame[key] = max(by_frame.get(key, -float("inf")), r[score_key])
    pos_top1 = [v for (lab, _, _), v in by_frame.items() if lab == "pos"]
    neg_top1 = [v for (lab, _, _), v in by_frame.items() if lab == "neg"]
    if not pos_top1 or not neg_top1:
        return float("nan")
    threshold = max(neg_top1)
    return float(sum(1 for v in pos_top1 if v > threshold) / len(pos_top1))


def evaluate(rows, score_key):
    cat_means = {}
    for bucket in ["pos_easy", "pos_hard", "neg_easy", "neg_hard"]:
        label, category = bucket.split("_")
        scores = [r[score_key] for r in rows
                  if r["label"] == label and r["category"] == category]
        cat_means[bucket] = float(np.mean(scores)) if scores else None

    pos = [r[score_key] for r in rows if r["label"] == "pos"]
    neg = [r[score_key] for r in rows if r["label"] == "neg"]
    pos_mean = float(np.mean(pos)) if pos else None
    neg_mean = float(np.mean(neg)) if neg else None
    sep = (pos_mean - neg_mean) if (pos_mean is not None and neg_mean is not None) else None

    return {
        **cat_means,
        "pos_mean": pos_mean,
        "neg_mean": neg_mean,
        "separation": sep,
        "auc": auc(pos, neg),
        "top1": top1_correctness(rows, score_key),
    }


brand_metrics = evaluate(results, "combined_score")
ovod_metrics = evaluate(results, "ovod_score")

# Print head-to-head comparison
print()
print(f"Eval set: {len(results)} {OVOD_MODEL} detections")
print()
header = f"{'metric':<14}  {'combined_brand':>15}  {'owlv2_text_score':>17}  {'delta':>8}"
print(header)
print("-" * len(header))
for k in ["pos_easy", "pos_hard", "neg_easy", "neg_hard",
          "pos_mean", "neg_mean", "separation", "auc", "top1"]:
    bv = brand_metrics[k]
    ov = ovod_metrics[k]
    if bv is None or ov is None:
        delta_str = "n/a"
        bv_str = f"{bv:.3f}" if bv is not None else "n/a"
        ov_str = f"{ov:.3f}" if ov is not None else "n/a"
    else:
        delta = bv - ov
        delta_str = f"{delta:+.3f}"
        bv_str = f"{bv:.3f}"
        ov_str = f"{ov:.3f}"
    print(f"{k:<14}  {bv_str:>15}  {ov_str:>17}  {delta_str:>8}")

# Save outputs
results_path = OUTDIR / "brand_results.jsonl"
summary_path = OUTDIR / "brand_summary.json"

with open(results_path, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

summary = {
    "ovod_model": OVOD_MODEL,
    "n_detections": len(results),
    "combined_brand": brand_metrics,
    "owlv2_text_score": ovod_metrics,
}
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print()
print("wrote", results_path)
print("wrote", summary_path)